In [22]:
# Expected Outcome

# A function named scrape_books that takes two parameters: min_rating and max_price. The function should scrape book data from the "Books to Scrape" website and return a pandas DataFrame with the following columns:

# Expected Outcome

# A function named scrape_books that takes two parameters: min_rating and max_price.
# The function should return a DataFrame with the following columns:
# UPC: The Universal Product Code (UPC) of the book.
# Title: The title of the book.
# Price (£): The price of the book in pounds.
# Rating: The rating of the book (1-5 stars).
# Genre: The genre of the book.
# Availability: Whether the book is in stock or not.
# Description: A brief description or product description of the book (if available).
# You will execute this script to scrape data for books with a minimum rating of 4.0 and above and a maximum price of £20.

# Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

In [ ]:
import os 
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time # para hacer una pausa al pedir datos y no se cuelgue por excesivas repeticiones

# Comprobacion conexion 
url = "https://books.toscrape.com/"
response = requests.get(url)
print(response) #para ver si estaba ok
print("---------------------------------------------")
print(response.headers) #para ver las cabeceras de la conexion
print("---------------------------------------------")


<Response [200]>
---------------------------------------------
{'Date': 'Mon, 25 May 2026 07:19:23 GMT', 'Content-Type': 'text/html', 'Content-Length': '51294', 'Connection': 'keep-alive', 'Last-Modified': 'Wed, 08 Feb 2023 21:02:32 GMT', 'ETag': '"63e40de8-c85e"', 'Accept-Ranges': 'bytes', 'Strict-Transport-Security': 'max-age=0; includeSubDomains; preload'}
---------------------------------------------


In [24]:
soup = BeautifulSoup(response.content, "html.parser")

In [25]:
# Detectar paginacion
base = "https://books.toscrape.com/"

posibles_urls = [
    base + "?page=1",
    base + "#page1",
    base + "page-1.html",
    base + "catalogue/page-1.html",
    base + "catalogue/?page=1",
    base + "catalogue/?offset=0",
]

for url in posibles_urls:
    r = requests.get(url)
    print(url, "→", r.status_code)
    


https://books.toscrape.com/?page=1 → 200
https://books.toscrape.com/#page1 → 200
https://books.toscrape.com/page-1.html → 404
https://books.toscrape.com/catalogue/page-1.html → 200
https://books.toscrape.com/catalogue/?page=1 → 403
https://books.toscrape.com/catalogue/?offset=0 → 403


In [26]:
# Pruebo las que salen 200 si tiene paginas. Si lase 0 no es valida aunque de 200

In [27]:

for i in range(1, 100):
    url = f"https://books.toscrape.com/catalogue/page={i}.html"
    r = requests.get(url)
    if r.status_code != 200:
        print("Última página válida:", i-1)
        break


Última página válida: 0


In [28]:
# Pruebo las que salen 200 si tiene paginas --> page{i}.html
for i in range(1, 100):
    url = f"https://books.toscrape.com/catalogue/page{i}.html"
    r = requests.get(url)
    if r.status_code != 200:
        print("Última página válida:", i-1)
        break

Última página válida: 0


In [29]:
# Pruebo las que salen 200 si tiene paginas --> page-{i}.html 
for i in range(1, 100):
    url = f"https://books.toscrape.com/catalogue/page-{i}.html"
    r = requests.get(url)
    if r.status_code != 200:
        print("Última página válida:", i-1)
        numero_paginas = i-1
        break

Última página válida: 50


Quiero guardar todos esos html y asi cada ve zwq me v

In [30]:


# 1. Crear carpeta donde guardar los HTML

os.makedirs("html_libros", exist_ok=True)


# 2. Obtener todas las URLs de los libros

def get_book_urls():
    urls = []
    base = "https://books.toscrape.com/catalogue/"

    for page in range(1, numero_paginas+1):
        url = f"{base}page-{page}.html"
        r = requests.get(url)
        soup = BeautifulSoup(r.content, "html.parser")

        books = soup.find_all("article", class_="product_pod")

        for book in books:
            relative = book.h3.a["href"]
            clean = relative.replace("../", "")   # limpia cualquier ../
            absolute = base + clean
            urls.append(absolute)

    return urls


# 3. Descargar todos los HTML y guardarlos en la carpeta

def descargar_html_libros():
    urls = get_book_urls()

    for url in urls:
        # Nombre del archivo basado en la carpeta del libro
        nombre_libro = url.split("/")[-2] + ".html"
        ruta = os.path.join("html_libros", nombre_libro)

        # Si ya existe, no lo descargamos
        if os.path.exists(ruta):
            continue

        print("Descargando:", url)
        r = requests.get(url)

        with open(ruta, "w", encoding="utf-8") as f:
            f.write(r.text)

        time.sleep(0.1) 

    print("✔ Descarga completada. Todos los HTML están en html_libros/")

# Ejecutar una vez
descargar_html_libros()


✔ Descarga completada. Todos los HTML están en html_libros/


In [31]:
# ahora que ya tenemos las url es facil sacar lo que queramos
# A function named scrape_books that takes two parameters: min_rating and max_price.
# The function should return a DataFrame with the following columns:
# UPC: The Universal Product Code (UPC) of the book.
# Title: The title of the book.
# Price (£): The price of the book in pounds.
# Rating: The rating of the book (1-5 stars).
# Genre: The genre of the book.
# Availability: Whether the book is in stock or not.
# Description: A brief description or product description of the book (if available).
def scrape_books(min_rating=0, max_price=999):
    # Listas donde guardaremos los datos
    upcs = []
    titles = []
    prices = []
    ratings = []
    genres = []
    availabilities = []
    descriptions = []

    # 1. Obtener todas las URLs de los libros
    base = "https://books.toscrape.com/catalogue/"
    urls = []

    for page in range(1, numero_paginas+1):
        url = f"{base}page-{page}.html"
        r = requests.get(url)
        soup = BeautifulSoup(r.content, "html.parser")

        books = soup.find_all("article", class_="product_pod")

        for book in books:
            relative = book.h3.a["href"]              # "../../../xxx/index.html"
            clean = relative.replace("../", "") # "xxx/index.html"
            absolute = base + clean                   # URL completa
            urls.append(absolute)

    # 2. Recorrer cada libro y extraer datos
    for url in urls:
        r = requests.get(url)
        soup = BeautifulSoup(r.content, "html.parser")

        # Título
        title = soup.find("h1").text

        # UPC
        table = soup.find("table", class_="table table-striped")
        upc = table.find("td").text

        # Precio
        price = float(soup.find("p", class_="price_color").text[2:])

        # Rating
        rating_text = soup.find("p", class_="star-rating")["class"][1]
        rating_num = ["Zero","One","Two","Three","Four","Five"].index(rating_text)

        # Filtros
        if rating_num < min_rating:
            continue
        if price > max_price:
            continue

        # Género
        genre = soup.find("ul", class_="breadcrumb").find_all("li")[2].text.strip()

        # Disponibilidad
        availability = soup.find("p", class_="instock availability").text.strip()

        # Descripción
        desc_tag = soup.find("div", id="product_description")
        description = desc_tag.find_next("p").text if desc_tag else None

        # Guardar en listas
        upcs.append(upc)
        titles.append(title)
        prices.append(price)
        ratings.append(rating_num)
        genres.append(genre)
        availabilities.append(availability)
        descriptions.append(description)

    # 3. Crear DataFrame
    df = pd.DataFrame({
        "UPC": upcs,
        "Title": titles,
        "Price (£)": prices,
        "Rating": ratings,
        "Genre": genres,
        "Availability": availabilities,
        "Description": descriptions
    })

    return df  

In [32]:
df = scrape_books(min_rating=4, max_price=20)
df


,UPC,Title,Price (£),Rating,Genre,Availability,Description
0,e00eb4fd7b871a48,Sharp Objects,7.82,4,Mystery,In stock (20 available),"WICKED above her hipbone, GIRL across her hear..."
1,4165285e1663650f,Sapiens: A Brief History of Humankind,4.23,5,History,In stock (20 available),From a renowned historian comes a groundbreaki...
2,2597b5a345f45e1b,The Dirty Little Secrets of Getting Your Dream...,3.34,4,Business,In stock (19 available),Drawing on his extensive experience evaluating...
3,e10e1e165dc8be4a,The Boys in the Boat: Nine Americans and Their...,2.60,4,Default,In stock (19 available),For readers of Laura Hillenbrand's Seabiscuit ...
4,30a7f60cd76ca58c,Shakespeare's Sonnets,0.66,4,Poetry,In stock (19 available),This book is an important and complete collect...
...,...,...,...,...,...,...,...
370,abc0b15f2c907ff0,Bounty (Colorado Mountain #7),7.26,4,Romance,In stock (1 available),Justice Lonesome has enjoyed a life of bounty....
371,099fae4a0705d63b,"Bleach, Vol. 1: Strawberry and the Soul Reaper...",4.65,5,Sequential Art,In stock (1 available),"Hot-tempered 15-year-old Ichigo Kurosaki, the ..."
372,bfd5e1701c862ac3,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",7.06,4,Sequential Art,In stock (1 available),High school student Kei Nagai is struck dead i...
373,19fec36a1dfb4c16,A Spy's Devotion (The Regency Spies of London #1),6.97,5,Historical Fiction,In stock (1 available),"In England’s Regency era, manners and elegance..."
